In [ ]:
# basic imports 
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
# transformators 
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
# models 
from sklearn.linear_model import LinearRegression, Ridge

'--------------------------------------------'
# data 
# parameters 
n = 50

# creating data
X_dom = np.linspace(-2, 3, 50)
X = np.random.uniform(-2, 3, n)
e = np.random.normal(size=n)
y = 1 + 2*X**2 + e 
# TODO ~ when we resize X first we are getting size 50x50 

# Task 1

Create a data (with n=50 datapoints) according to the formula $y=1+2x^2+e$, where $x\in [-2,3]$ and $e\sim \mathcal{N}(0,1)$. Fit the model of the form (use `PolynomialFeatures`):
- $y=\beta_0+\beta_1x$
- $y=\beta_0+\beta_1x+\beta_2x^2$
- $y=\beta_0+\beta_1x+\beta_2x^2+\beta_3 x^3+ \ldots +\beta_{20} x^{20}$

Prepare the plots with true and fitted models.



### Solution

#### Method  
We iterated through a set of predefined polynomial degrees. For each degree:  
- a model was created using polynomial features of the corresponding degree,  
- the model was trained on the available data,  
- and the resulting fit was visualized to assess model behavior.  

#### Results and Interpretation  
This experiment clearly illustrates the concepts of **underfitting** and **overfitting**.  
- When the polynomial degree is too **low**, the model is too simple to capture the underlying pattern in the data. As a result, predictions are inaccurate — this is a case of **underfitting**.  
- When the degree is too **high**, the model becomes overly complex and starts fitting not only the true signal but also the noise in the data. This leads to poor generalization and unstable predictions — a typical example of **overfitting**.

The exercise effectively demonstrates how model complexity impacts performance and highlights the importance of selecting an appropriate polynomial degree to balance bias and variance.





In [ ]:
# creating polynomial models 
for i in (1, 2, 20):
    # training model 
    Poly = PolynomialFeatures(i)
    train_data = Poly.fit_transform(X.reshape(-1, 1))
    model = LinearRegression()
    model.fit(train_data, y)
    # plot values
    X_test_domain = Poly.transform(X_dom.reshape(-1, 1))
    val = model.predict(X_test_domain)
    # plot
    plt.plot(X_dom, val, color='red')
    plt.scatter(X, y)
    plt.ylim(-1 + y.min(), 1 + y.max())
    plt.show()
    print(model.score(train_data, y))


# Task 2

Using the data from Task 1, find the best polynomial degree using cross-validation. Visualise the results.

# Task 3.
Use the above dataset. Fit polynomial ridge regression (degree 20) with lambda equal 1 and 100 (in the penalty function take into account all coeaficionts). Use scaling.

Draw the data and all curves on one figure.


Compare the new curves on the graph with a polynomial regression of degree 20.


### Solution

#### Method  
The method is the same as in the previous exercise.  
This time, however, we fix the polynomial degree at **20** and apply **Ridge regression** with different regularization strengths:  
- λ = 1  
- λ = 100  

#### Results and Interpretation  
We observe how Ridge regression influences the model by **reducing the variance of the coefficients**, even though the model itself remains complex.  
- The regularization term helps to **stabilize the model**, making the coefficients smaller and more balanced.  
- As a result, although the model still exhibits some overfitting due to high polynomial degree, it becomes **more reliable for prediction**, especially when using stronger regularization (e.g., λ = 100).  

This experiment demonstrates how Ridge regression can mitigate the negative effects of overfitting by controlling coefficient magnitudes, thus improving generalization.


In [ ]:
# parameters
deg = 20 

 # scalling na polynomial czy na wejściowe dane ??? bo 


scaler = StandardScaler()
StandardScaler.fit_transform
# training polynomomial
poly_3 = PolynomialFeatures(deg)
X_poly_3 = poly_3.fit_transform(X.reshape(-1, 1))

# exercise 
for a in (1, 100):
    # creating model
    model = make_pipeline(
        PolynomialFeatures(3),
        StandardScaler(),
        Ridge(a)
        )

    # training model
    model.fit(X_poly_3, y)
    # creating plot
    pred = model.predict(poly_3.transform(X_dom.reshape(-1, 1)))
    plt.plot(X_dom, pred, label=f'Model : $\lambda = {a}$')
    print(model.score(X_poly_3, y))

plt.scatter(X, y)
plt.ylim(y.min() - 1, y.max() + 1)
plt.legend()
plt.show()

# printing plot 


# Task 4.
In Task 3 find the best parameter C by using cross-validation.

Compare the new curves on the graph with a polynomial regression of degree 20.

# Task 5
Calculate (manually) Rigde regression beta parameters for 
$\lambda=1$,
$x_1=[0,1,2,3]$,
$x_2=[0,0,1,1]$,
$y=[2,2,3]$.

#### Data:
Dimension of $y$ is $3$ and dimension of $x$ is $4$. In order to make dimension similiar i am dropping values from features on last index. 


#### Method
To find Ridge regression parameters we need to optimilize cost function of form:
 
$(y - X\beta)^t(y - X\beta) + \lambda \beta^t \beta$

After solving minimization problem we obtain that our estimator should be of form: 

$\hat\beta = \left(X^tX + \lambda I\right)^{-1} X^ty$

To calculate $\hat\beta$ including intercept we adding ones to our feature column $X$, and we adjusting $I$ so $I_{0, 0} = 0$. It is because adding weight penalty for $\beta_0$ would make procedure dependent on origin. (The elements of statistical learning chapter 3 page 59) 


#### Solution

We can see that manually calculating ridge regression gave ous same result as using built in method fit on object Ridge from sklearn.linear_model


In [ ]:
# creating X, y variables
c = 1
X_1 = np.array([
    [0, 1, 2],
    [0, 0, 1]
]).reshape(3, 2)
# adding intercept 
X = np.concat((np.ones((X_1.shape[0], 1)), X_1), axis=1)
y = np.array([2, 2, 3]).reshape(-1, 1)

# finding b parameter WARNING we need to not evaluate on eye our identity object 
I = np.eye(X.shape[1]) 
I[0, 0] = 0
b = np.linalg.solve((X.T @ X + c * I), X.T @ y)

# finding b from build in regression 
lm = Ridge(c, fit_intercept=True)
lm.fit(X_1, y)
print(np.concat((lm.intercept_, lm.coef_)), '\n', b.reshape(-1, ))


